# 04 — Ulasan eksperimen dan keputusan submission

Notebook ini meninjau hasil eksekusi sesi sebelumnya, **bukan melatih ulang model**. Artefak model sementara tidak tersimpan setelah lingkungan sesi berganti. Hasil ini bukan hasil `scripts/train.py` dan bukan skor Kaggle. Target 0,65 belum tercapai. Sel kode dijalankan secara berurutan melalui IPython dalam proses lokal; kernel Jupyter terpisah tidak tersedia pada lingkungan verifikasi.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if Path('scripts').exists() else Path.cwd().parent
history = json.loads((ROOT/'reports'/'experiment_history.json').read_text())
print(history['provenance'])
results = pd.DataFrame(history['experiments'])
results['blend_delta_vs_incumbent'] = results.best_blend_selection_macro_f1 - history['baseline_selection_macro_f1']
display(results)
assert (results.blend_delta_vs_incumbent < 0).all()

Recovered from actual earlier-session execution outputs; temporary trained models and per-row new experiment predictions were not retained. These are not results of scripts/train.py.


,name,leaves,trees,class_weight_power,standalone_selection_macro_f1,best_blend_selection_macro_f1,blend_delta_vs_incumbent
0,te_lgb_63,63,650,0.00,0.584729,0.599438,-0.000645
1,te_lgb_63_balanced,63,650,0.40,0.584294,0.599429,-0.000653
2,te_lgb_127,127,750,0.25,0.582991,0.599071,-0.001012
3,te_lgb_31_ova,31,850,0.25,0.581765,0.599380,-0.000703
4,te_lgb_coarse,63,850,0.30,0.573951,0.599190,-0.000893
5,te_lgb_127_smooth,127,1000,0.40,0.581998,0.598819,-0.001264
6,te_lgb_31_long,31,1300,0.25,0.582419,0.598863,-0.001220
7,te_lgb_127_ova,127,750,0.35,0.582816,0.599390,-0.000693


## Confusion matrix dan metrik kelas

Baris = aktual; kolom = prediksi. Sebanyak 60.000 baris berasal dari confirmation historis dengan adaptive selection. Evaluasi ini bukan test independen baru.

In [2]:
cm = np.asarray(history['baseline_confusion_matrix'])
assert cm.sum() == 60000
precision = np.diag(cm) / cm.sum(axis=0)
recall = np.diag(cm) / cm.sum(axis=1)
f1 = 2 * precision * recall / (precision + recall)
display(pd.DataFrame(cm, index=range(1,6), columns=range(1,6)))
display(pd.DataFrame({'precision':precision, 'recall':recall, 'f1':f1, 'support':cm.sum(axis=1)},index=range(1,6)))
print('Recomputed incumbent Macro F1:', f1.mean())
assert np.isclose(f1.mean(), history['regional']['baseline_pooled_macro_f1'])

Recomputed incumbent Macro F1: 0.601198135554628


,1,2,3,4,5
1,4289,1267,350,182,117
2,1021,3411,1688,526,224
3,485,1780,5373,2367,735
4,192,573,2332,9058,2319
5,207,512,1386,4183,15423


,precision,recall,f1,support
1,0.692444,0.691217,0.691830,6205
2,0.452207,0.496507,0.473323,6870
3,0.482793,0.500279,0.491380,10740
4,0.555161,0.625812,0.588373,14474
5,0.819588,0.710377,0.761085,21711


Kelas 2 dan 3 memiliki F1 terendah. Kesalahan pada tingkat kerusakan berdekatan mendukung pengujian ordinal classification berikutnya. Delapan percobaan class weighting, kapasitas, smoothing, objective dan blend belum mengalahkan incumbent.

In [3]:
regional = history['regional']
folds = pd.DataFrame({'incumbent':regional['fold_baseline'], 'candidate':regional['fold_candidate']})
folds['delta'] = folds.candidate - folds.incumbent
display(folds)
print('Pooled incumbent:', regional['baseline_pooled_macro_f1'])
print('Pooled candidate:', regional['candidate_pooled_macro_f1'])
print('Delta:', regional['delta'])
print('Recorded bootstrap 95% CI:', regional['paired_group_bootstrap_95_ci'])
print('Promote candidate:', history['promote_candidate'])
assert regional['paired_group_bootstrap_95_ci'][0] < 0 < regional['paired_group_bootstrap_95_ci'][1]

Pooled incumbent: 0.6011981355546281
Pooled candidate: 0.6013795567737634
Delta: 0.0001814212191353
Recorded bootstrap 95% CI: [-0.0001672139789578808, 0.0005384330334488734]
Promote candidate: False


,incumbent,candidate,delta
0,0.603368,0.603377,0.000009
1,0.602887,0.602860,-0.000027
2,0.597165,0.597733,0.000569


## Keputusan: pertahankan submission incumbent

Kenaikan lokal +0,000181 belum meyakinkan: interval bootstrap mencakup nol dan satu fold menurun. Bootstrap bersyarat pada resep terpilih dan tidak mengoreksi riwayat adaptive tuning. Public/private score kandidat belum diverifikasi.

Lihat `reports/EXPERIMENT_RESULTS.md` untuk protokol, konfigurasi, batas reproduksi dan rencana eksperimen lanjutan. Notebook 01–03 harus dijalankan dengan data kompetisi untuk menghasilkan model dan CSV baru.